# M2-03: Research Agent MVP

## Контекст

Этот notebook реализует **Research Agent** — агентный подход к поиску актуальной информации для обогащения генерируемых материалов.

### Отличие от линейного поиска

| Подход | Линейный | Агентный (Research Agent) |
|--------|----------|---------------------------|
| Логика | N запросов → search → результаты | Итеративное погружение с адаптацией |
| Планирование | Статичное | Динамическое (adaptive planning) |
| Reasoning | Нет | Structured reasoning через Pydantic |
| Citations | Нет | Inline citations [1], [2], [3] |

### Workflow Integration

```
Input:
  - input_content (тема/вопрос)
  - handwritten_notes (опционально, передается AS IS)

Process:
  1. Generate plan (iteration 0)
  2. Iterate: reason → select action → execute
  3. Adapt plan if needed
  4. Create final report with citations

Output:
  - final_report (с inline citations [1], [2])
  - sources (dict[url, SourceData])
```

### MVP Scope

**Что реализуем:**
- Tavily web search only
- Structured reasoning через Pydantic schemas
- Adaptive planning
- Source management с citations
- LangGraph StateGraph с одним node

**Что НЕ реализуем:**
- RAG по Telegram
- Document search (PDF)
- HITL через interrupt (упрощенная версия)
- Интеграция с основным LearnFlow workflow

## 1. Setup and Environment

In [1]:
import os
import sys
from pathlib import Path
from typing import Any, Literal, Optional
from dotenv import load_dotenv

# Add project root to path
project_root = Path().cwd().parent.parent
sys.path.insert(0, str(project_root))

# Load environment variables
env_local = project_root / ".env.local"
env_file = project_root / ".env"

if env_local.exists():
    load_dotenv(env_local)
    print(f"✓ Loaded .env.local from {env_local}")
elif env_file.exists():
    load_dotenv(env_file)
    print(f"✓ Loaded .env from {env_file}")
else:
    print("⚠ No .env file found")

print(f"✓ Project root: {project_root}")

# Helper function to resolve API keys
def resolve_api_key(key_value: str) -> str:
    """
    Resolve API key from config value.
    If starts with $, read from environment variable.
    Otherwise, use the value directly.
    """
    if key_value.startswith("$"):
        env_var = key_value[1:]  # Remove $
        value = os.getenv(env_var)
        if not value:
            raise ValueError(f"Environment variable {env_var} not found")
        return value
    return key_value

print("✓ Helper functions defined")

✓ Loaded .env.local from /home/bbaron/dev/my_pet_projects/learnflow-ai/.env.local
✓ Project root: /home/bbaron/dev/my_pet_projects/learnflow-ai
✓ Helper functions defined


In [2]:
# Core imports
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.types import Command
from tavily import TavilyClient

print("✓ All imports successful")

✓ All imports successful


## 2. Configuration

In [3]:
# Configuration placeholders (will be overridden by M2-config.yaml)
# DO NOT set default values here - they will be loaded from config

# Agent Configuration - Educational Content Research
MAX_ITERATIONS = 8
MAX_SEARCHES = 6
MAX_EXTRACTIONS = 5

print("⚠ MODEL_NAME and TEMPERATURE will be loaded from M2-config.yaml")
print(f"✓ Max iterations: {MAX_ITERATIONS}")
print(f"✓ Max searches: {MAX_SEARCHES}")
print(f"✓ Max extractions: {MAX_EXTRACTIONS}")

⚠ MODEL_NAME and TEMPERATURE will be loaded from M2-config.yaml
✓ Max iterations: 8
✓ Max searches: 6
✓ Max extractions: 5


## 3. Data Models

### 3.1. SourceData Model

In [4]:
class SourceData(BaseModel):
    """Модель источника информации"""
    number: int = Field(description="Citation number [1], [2], [3]")
    title: str = Field(description="Заголовок источника")
    url: str = Field(description="URL источника")
    snippet: str = Field(default="", description="Короткий snippet из search")
    full_content: str = Field(default="", description="Полный контент страницы (опционально)")
    char_count: int = Field(default=0, description="Количество символов в full_content")

print("✓ SourceData model defined")

✓ SourceData model defined


### 3.2. ResearchState Model

In [5]:
class ResearchState(BaseModel):
    """LangGraph state for research agent"""

    # Input
    input_content: str = Field(description="User's topic or question")

    # Execution state
    iteration: int = Field(default=0, description="Current iteration number")
    agent_state: Literal["planning", "searching", "extracting", "synthesizing", "completed"] = Field(
        default="planning",
        description="Current agent state"
    )

    # Accumulated knowledge (dict for deduplication)
    sources: dict[str, SourceData] = Field(default_factory=dict, description="URL → SourceData mapping")

    # History for LLM context (agent's decision-making memory)
    decision_history: list[Any] = Field(default_factory=list, description="Agent's decision history for context between iterations")

    # Resource counters
    searches_used: int = Field(default=0, description="Number of searches performed")
    extractions_used: int = Field(default=0, description="Number of content extractions performed")

    # Results
    research_goal: str = Field(default="", description="Current research goal")
    final_report: Optional[str] = Field(default=None, description="Final report with citations")

print("✓ ResearchState model defined")

✓ ResearchState model defined


## 4. Tool Schemas (Pydantic)

Все решения агента через structured output (Pydantic schemas), НЕ function calling.

In [6]:
# ============================================================================
# DISCRIMINATED UNION PATTERN
# ============================================================================

from pydantic import create_model
from typing import Annotated, Type, TypeVar
from functools import reduce
import operator

T = TypeVar("T", bound=BaseModel)


class DiscriminatorMixin(BaseModel):
    """Adds discriminator field and excludes it from serialization"""
    tool_type: str = Field(..., description="Tool type identifier")

    def model_dump(self, *args, **kwargs):
        # Exclude tool_type from serialization (technical field for LLM only)
        exclude = kwargs.pop("exclude", set())
        exclude = exclude.union({"tool_type"})
        return super().model_dump(*args, exclude=exclude, **kwargs)


def wrap_with_discriminator(tool_class: Type[T], discriminator_value: str) -> Type[BaseModel]:
    """
    Creates a version of the tool with discriminator field.
    
    Args:
        tool_class: Original tool class
        discriminator_value: Value for discriminator field
    
    Returns:
        New class with DiscriminatorMixin
    """
    return create_model(
        f"D_{tool_class.__name__}",
        __base__=(tool_class, DiscriminatorMixin),  # Order matters!
        tool_type=(
            Literal[discriminator_value],
            Field(default=discriminator_value, description="Tool type")
        ),
    )


# ============================================================================
# TOOL SCHEMAS
# ============================================================================

class ReasoningTool(BaseModel):
    """Structured reasoning for decision transparency"""
    reasoning_steps: list[str] = Field(
        min_length=2, 
        max_length=3, 
        description="Key reasoning steps (2-3 concise points)"
    )
    current_situation: str = Field(
        max_length=300, 
        description="Current research situation summary"
    )
    enough_data: bool = Field(
        description=(
            "Sufficient data to support comprehensive EDUCATIONAL content generation. "
            "Consider: multiple authoritative sources, mix of theory and practice, "
            "enough material for thorough explanations."
        )
    )
    next_steps: list[str] = Field(
        min_length=1, 
        max_length=3, 
        description="Planned next actions (1-3)"
    )
    task_completed: bool = Field(description="Research task finished")


class WebSearchTool(BaseModel):
    """Web search via Tavily"""
    query: str = Field(description="Search query to find relevant information")
    max_results: int = Field(
        default=3, 
        ge=1, 
        le=5, 
        description="Number of results to return (1-5)"
    )


class ExtractPageContentTool(BaseModel):
    """Extract full page content from a source"""
    source_number: int = Field(
        ge=1, 
        description="Source citation number [1], [2], [3] to extract full content from"
    )
    reasoning: str = Field(description="Why this source is worth extracting")


class GeneratePlanTool(BaseModel):
    """Create initial research plan"""
    reasoning: str = Field(description="Rationale for the research plan")
    research_goal: str = Field(description="Clear, specific research objective")
    planned_steps: list[str] = Field(
        min_length=2, 
        max_length=5, 
        description="Planned research steps (2-5)"
    )
    search_strategies: list[str] = Field(
        min_length=1, 
        max_length=3, 
        description="Search strategies to employ (1-3)"
    )


class AdaptPlanTool(BaseModel):
    """Adapt research plan based on findings"""
    reasoning: str = Field(description="Why plan adaptation is needed")
    original_goal: str = Field(description="Original research goal")
    new_goal: str = Field(description="Updated research goal")
    plan_changes: list[str] = Field(
        min_length=1, 
        max_length=3, 
        description="Specific changes to the plan (1-3)"
    )
    next_steps: list[str] = Field(
        min_length=1, 
        max_length=3, 
        description="Next actions under new plan (1-3)"
    )


class CreateReportTool(BaseModel):
    """
    Create research summary for educational content generation.
    
    The report should provide information-dense material that helps the downstream
    system generate comprehensive learning content. Focus on facts, explanations,
    examples, and best practices - not narrative polish.
    """
    content: str = Field(
        description=(
            "Information-dense research summary with inline citations [1], [2], [3]. "
            "Include: key concepts, practical examples, current best practices, "
            "technical details for teaching. Focus on educational value."
        )
    )


class FinalAnswerTool(BaseModel):
    """
    Complete research without creating a report.
    
    Use when:
    - Topic doesn't require web search (classical theory, math)
    - Search failed to find relevant results
    - Request is outside capabilities
    """
    message: str = Field(description="Final message explaining completion reason")


# ============================================================================
# WRAPPER MODEL (STUB)
# ============================================================================

class NextStepToolStub(BaseModel):
    """Stub for IDE autocomplete. Real model created via build_next_step_tools()"""
    action: T  # Generic placeholder for the selected tool


# ============================================================================
# DYNAMIC UNION BUILDER
# ============================================================================

def build_next_step_tools(tools: list[tuple[Type[T], str]]) -> Type[NextStepToolStub]:
    """
    Creates Pydantic model with discriminated union of available tools.
    
    Pattern from DISCRIMINATED_UNION_PATTERN.md.
    Pydantic automatically recognizes discriminated union through Literal fields.
    
    Args:
        tools: List of (tool_class, discriminator_id) tuples
    
    Returns:
        Pydantic model with 'action' field containing discriminated union
    """
    if len(tools) == 1:
        # Single tool - wrap and return in stub
        wrapped = wrap_with_discriminator(tools[0][0], tools[0][1])
        return create_model(
            "NextStepTools",
            __base__=(NextStepToolStub,),
            action=(wrapped, Field(description="Selected tool action")),
        )

    # Wrap all tools with discriminator
    wrapped = [wrap_with_discriminator(cls, disc_id) for cls, disc_id in tools]
    
    # Create Union: A | B | C | D
    union = reduce(operator.or_, wrapped)
    
    # Pydantic automatically recognizes discriminated union through Literal fields
    # NO explicit discriminator= needed (causes OpenAI oneOf error)
    union_annotated = Annotated[union, Field()]
    
    # Create wrapper model with union as 'action' field
    return create_model(
        "NextStepTools",
        __base__=(NextStepToolStub,),
        action=(union_annotated, Field(description="Selected tool action")),
    )


def get_available_tools(state: ResearchState) -> Type[NextStepToolStub]:
    """
    Returns dynamic Pydantic model with available tools.
    
    Resource management via constrained decoding.
    """
    # Forced completion at MAX_ITERATIONS
    if state.iteration >= MAX_ITERATIONS:
        return build_next_step_tools([
            (CreateReportTool, "create_report"),
            (FinalAnswerTool, "final_answer")
        ])

    # Planning phase (iteration 0)
    if state.iteration == 0:
        return build_next_step_tools([
            (GeneratePlanTool, "generate_plan"),
            (ReasoningTool, "reasoning")
        ])

    # Build list of available tools
    available = [
        (ReasoningTool, "reasoning"),
        (CreateReportTool, "create_report"),
        (FinalAnswerTool, "final_answer"),
        (AdaptPlanTool, "adapt_plan")
    ]

    # Add conditional tools
    if state.searches_used < MAX_SEARCHES:
        available.append((WebSearchTool, "web_search"))

    if state.extractions_used < MAX_EXTRACTIONS and len(state.sources) > 0:
        available.append((ExtractPageContentTool, "extract_content"))

    return build_next_step_tools(available)


print("✓ Tool schemas defined with discriminated unions")
print("✓ NextStepToolStub wrapper model defined")
print("✓ build_next_step_tools() function defined (implicit discriminator)")
print("✓ get_available_tools() function defined")

✓ Tool schemas defined with discriminated unions
✓ NextStepToolStub wrapper model defined
✓ build_next_step_tools() function defined (implicit discriminator)
✓ get_available_tools() function defined


## 5. Tavily Integration

Функции для работы с Tavily API.

In [7]:
# Tavily client will be initialized after loading M2-config.yaml
# See cell "LOAD CONFIGURATION" below
print("⚠ Tavily client initialization moved to LOAD CONFIGURATION section")

⚠ Tavily client initialization moved to LOAD CONFIGURATION section


In [8]:
def execute_web_search(
    tool: WebSearchTool,
    state: ResearchState
) -> dict[str, SourceData]:
    """
    Выполняет web search через Tavily и возвращает новые источники.
    
    Args:
        tool: WebSearchTool с query и max_results
        state: Текущее состояние (для нумерации sources)
    
    Returns:
        dict[url, SourceData] с новыми источниками
    """
    print(f"\n🔍 Searching: '{tool.query}'")
    
    # Tavily search
    response = tavily_client.search(
        query=tool.query,
        max_results=tool.max_results,
        search_depth="basic"
    )
    
    results = response.get("results", [])
    print(f"✓ Found {len(results)} results")
    
    # Create SourceData objects
    new_sources = {}
    next_number = len(state.sources) + 1
    
    for result in results:
        url = result["url"]
        
        # Skip if already exists
        if url in state.sources:
            print(f"  ⊘ Duplicate: {url}")
            continue
        
        source = SourceData(
            number=next_number,
            title=result.get("title", "No title"),
            url=url,
            snippet=result.get("content", "")[:200],  # First 200 chars
            char_count=len(result.get("content", ""))
        )
        
        new_sources[url] = source
        print(f"  [{next_number}] {source.title[:60]}...")
        next_number += 1
    
    return new_sources


print("✓ execute_web_search() function defined")

✓ execute_web_search() function defined


## 5.5. System Prompts

Вынесенные промпты для ResearchAgentNode (hardcoded для MVP).

In [9]:
# ============================================================================
# SYSTEM PROMPT
# ============================================================================

SYSTEM_PROMPT = """<role>
You are a research assistant preparing source material for educational content generation.
Your research will be used by a downstream system to create comprehensive learning materials.
</role>

<context>
Your output feeds into an educational content generator that creates:
- Structured learning materials (theory + practice)
- Examples and exercises
- Explanations suitable for students

The generator needs rich, authoritative information to work with.
</context>

<guidelines>
- Prioritize EDUCATIONAL value: explanations, examples, practical applications
- Seek authoritative sources (official docs, academic resources, recognized experts)
- Focus on depth: collect material that supports thorough understanding
- Include multiple angles: conceptual explanations, practical examples, common pitfalls
- Extract content that helps TEACH, not just inform
- Classical topics (pure math, established theory) may not need web search - explain and exit early
- Iterate through research: plan → search → extract → synthesize → report
- Adapt your research plan if findings contradict assumptions or reveal new directions
- Use structured reasoning to make decisions transparent
- Collect sources and assign citation numbers [1], [2], [3] as you find them
- Extract full content only from sources that seem highly relevant
- Create information-dense reports with inline citations
</guidelines>

<constraints>
- All decisions must use the available tool schemas
- Citations must reference source numbers in your collected sources
- Be efficient with searches and extractions - they are limited resources
</constraints>"""

print("✓ System prompt defined")

✓ System prompt defined


In [10]:
def execute_extract_content(
    tool: ExtractPageContentTool,
    state: ResearchState
) -> dict[str, SourceData]:
    """
    Извлекает полный контент одной страницы по номеру источника.

    Args:
        tool: ExtractPageContentTool с source_number
        state: Текущее состояние (для поиска source)

    Returns:
        dict[url, SourceData] с обновленным источником
    """
    print(f"\n📄 Extracting content from source [{tool.source_number}]")
    print(f"   Reasoning: {tool.reasoning}")

    # Найти source по номеру
    target_source = None
    target_url = None

    for url, source in state.sources.items():
        if source.number == tool.source_number:
            target_source = source
            target_url = url
            break

    if not target_source:
        print(f"  ✗ Source [{tool.source_number}] not found in state")
        return {}

    print(f"   URL: {target_url}")

    # Tavily extract
    try:
        response = tavily_client.extract(urls=[target_url])
        content = response.get("results", [])[0].get("raw_content", "")

        # Update existing source
        updated_source = target_source.model_copy()
        updated_source.full_content = content[:5000]  # Limit to 5000 chars
        updated_source.char_count = len(content)

        print(f"  ✓ Extracted {updated_source.char_count} chars (stored: {len(updated_source.full_content)})")

        return {target_url: updated_source}

    except Exception as e:
        print(f"  ✗ Error extracting {target_url}: {e}")
        return {}


print("✓ execute_extract_content() function defined")

✓ execute_extract_content() function defined


## 6. ResearchAgentNode Implementation

Основной узел агента с логикой планирования, поиска и отчетности.

In [11]:
class ResearchAgentNode:
    """Research Agent node for LangGraph workflow"""
    
    def __init__(self, model_name: str, temperature: float, api_key: str, provider_config: dict = None):
        # Build LLM kwargs
        llm_kwargs = {
            "model": model_name,
            "temperature": temperature,
            "api_key": api_key,
        }
        
        # Add base_url if configured
        if provider_config and provider_config.get("base_url"):
            llm_kwargs["base_url"] = provider_config["base_url"]
            print(f"  ✓ Using custom base_url: {provider_config['base_url']}")
        
        self.model = ChatOpenAI(**llm_kwargs)
        print(f"  ✓ LLM initialized: {model_name} (temp={temperature})")
    
    def _build_messages(self, state: ResearchState) -> list:
        """
        Build messages for LLM context.
        
        All state is conveyed through conversation history, not system prompt.
        System prompt contains only static guidelines.
        """
        messages = [SystemMessage(content=SYSTEM_PROMPT)]
        
        # Add initial user query if this is first message
        if not state.decision_history:
            messages.append(HumanMessage(content=f"Research topic: {state.input_content}"))
        
        # Add conversation history (AIMessage with tool_calls, ToolMessage with results)
        messages.extend(state.decision_history)
        
        return messages
    
    async def _select_action(self, state: ResearchState, config: dict):
        """
        Select next action using structured output.
        
        Returns:
            tuple: (action, ai_message_with_tool_call) for emulating function calling
        """
        print(f"\n🤔 Selecting action (iteration {state.iteration})...")
        
        messages = self._build_messages(state)
        
        # Get available tools based on current state
        available_tools_model = get_available_tools(state)
        llm = self.model.with_structured_output(available_tools_model)
        
        # Model returns wrapper with .action field (structured output)
        response = await llm.ainvoke(messages)
        action = response.action
        
        # Get tool type for logging
        tool_type = action.model_dump().get("tool_type", type(action).__name__)
        print(f"✓ Selected: {tool_type}")
        
        # ⭐ EMULATE tool_calls for OpenAI conversation history
        # LangChain format for tool_calls (different from raw OpenAI format)
        ai_message_with_tool_call = AIMessage(
            content="",  # Could add reasoning text here
            tool_calls=[
                {
                    "name": type(action).__name__,  # Tool class name
                    "args": action.model_dump(exclude={"tool_type"}),  # Tool parameters as dict (exclude discriminator)
                    "id": f"call_{state.iteration}",  # Unique ID per iteration
                }
            ],
        )
        
        return action, ai_message_with_tool_call
    
    def _format_tool_result(self, action, execution_result: dict) -> str:
        """
        Format tool execution result for ToolMessage.
        
        Returns human-readable summary of what happened.
        """
        if isinstance(action, WebSearchTool):
            sources_count = len(execution_result.get("sources", {}))
            return f"Found {sources_count} new sources for query: '{action.query}'"
        
        elif isinstance(action, ExtractPageContentTool):
            sources = execution_result.get("sources", {})
            if sources:
                source = list(sources.values())[0]
                return f"Extracted {source.char_count} chars from source [{action.source_number}]"
            return f"Failed to extract from source [{action.source_number}]"
        
        elif isinstance(action, GeneratePlanTool):
            return f"Research plan created: {action.research_goal}"
        
        elif isinstance(action, AdaptPlanTool):
            return f"Plan adapted: {action.new_goal}"
        
        elif isinstance(action, ReasoningTool):
            steps = "\n".join([f"  - {s}" for s in action.reasoning_steps])
            return f"Reasoning:\n{steps}\nEnough data: {action.enough_data}"
        
        elif isinstance(action, CreateReportTool):
            return f"Report created ({len(action.content)} chars)"
        
        elif isinstance(action, FinalAnswerTool):
            return f"Research completed: {action.message}"
        
        return "Action executed"
    
    async def _execute_action(
        self,
        action,
        ai_message_with_tool_call: AIMessage,
        state: ResearchState
    ) -> dict:
        """
        Execute selected action and return state updates.
        
        Args:
            action: Selected tool instance
            ai_message_with_tool_call: AIMessage with tool_calls (for history)
            state: Current state
        
        Returns:
            dict: State updates including decision_history with AIMessage + ToolMessage
        """
        updates = {}
        execution_result = {}
        
        # Execute based on action type
        if isinstance(action, WebSearchTool):
            new_sources = execute_web_search(action, state)
            updates["sources"] = {**state.sources, **new_sources}
            updates["searches_used"] = state.searches_used + 1
            execution_result["sources"] = new_sources
        
        elif isinstance(action, ExtractPageContentTool):
            updated_sources = execute_extract_content(action, state)
            updates["sources"] = {**state.sources, **updated_sources}
            updates["extractions_used"] = state.extractions_used + 1
            execution_result["sources"] = updated_sources
        
        elif isinstance(action, GeneratePlanTool):
            updates["research_goal"] = action.research_goal
            print(f"✓ Research goal set: {action.research_goal}")
        
        elif isinstance(action, AdaptPlanTool):
            updates["research_goal"] = action.new_goal
            print(f"✓ Plan adapted: {action.new_goal}")
        
        elif isinstance(action, CreateReportTool):
            updates["final_report"] = action.content
            updates["agent_state"] = "completed"
            print(f"✓ Report created: {len(action.content)} chars")
        
        elif isinstance(action, ReasoningTool):
            print(f"💭 Reasoning:")
            for step in action.reasoning_steps:
                print(f"  - {step}")
            print(f"  Enough data: {action.enough_data}")
            print(f"  Task completed: {action.task_completed}")
        
        elif isinstance(action, FinalAnswerTool):
            updates["agent_state"] = "completed"
            print(f"✓ Final answer: {action.message}")
        
        # ⭐ Build conversation history with emulated tool_calls
        # Extract tool_call_id from ai_message_with_tool_call
        tool_call_id = ai_message_with_tool_call.tool_calls[0]["id"]
        
        # Create ToolMessage with matching tool_call_id
        tool_message = ToolMessage(
            content=self._format_tool_result(action, execution_result),
            tool_call_id=tool_call_id  # Must match ID from AIMessage.tool_calls
        )
        
        # Update history: AIMessage (with tool_calls) + ToolMessage (with tool_call_id)
        updates["decision_history"] = state.decision_history + [
            ai_message_with_tool_call,  # 1. Assistant message with tool_calls
            tool_message                 # 2. Tool result message
        ]
        
        return updates
    
    def _should_continue(self, state: ResearchState) -> bool:
        """Check if agent should continue iterations"""
        if state.agent_state == "completed":
            return False
        if state.iteration >= MAX_ITERATIONS:
            return False
        if state.searches_used >= MAX_SEARCHES and state.extractions_used >= MAX_EXTRACTIONS:
            # Can still continue if can create report or reason
            return True
        return True
    
    async def __call__(self, state: ResearchState, config: dict = None) -> Command:
        """Main node execution logic"""
        config = config or {}
        
        # Iteration loop
        if self._should_continue(state):
            # Get action and ai_message with tool_calls
            action, ai_message = await self._select_action(state, config)
            
            # Execute action with ai_message for proper history
            updates = await self._execute_action(action, ai_message, state)
            updates["iteration"] = state.iteration + 1
            
            # Check if completed
            if updates.get("agent_state") == "completed":
                return Command(goto=END, update=updates)
            
            return Command(goto="research_agent", update=updates)
        
        # Forced finalization at max iterations (if no report yet)
        if not state.final_report:
            print("\n⚠ Max iterations reached, forcing report creation...")
            
            # Force CreateReportTool selection
            messages = self._build_messages(state)
            llm = self.model.with_structured_output(CreateReportTool)
            report_action = await llm.ainvoke(messages)
            
            # ⭐ Emulate tool_calls for forced report (LangChain format)
            ai_message = AIMessage(
                content="",
                tool_calls=[{
                    "name": "CreateReportTool",
                    "args": report_action.model_dump(),
                    "id": f"call_{state.iteration}_forced",
                }],
            )
            
            tool_message = ToolMessage(
                content=f"Report created ({len(report_action.content)} chars)",
                tool_call_id=f"call_{state.iteration}_forced",  # Must match!
            )
            
            return Command(
                goto=END,
                update={
                    "final_report": report_action.content,
                    "agent_state": "completed",
                    "decision_history": state.decision_history + [ai_message, tool_message]
                }
            )
        
        # Already completed
        return Command(goto=END, update={})


print("✓ ResearchAgentNode class defined")

✓ ResearchAgentNode class defined


## 7. LangGraph Workflow

Создание StateGraph с одним узлом.

In [12]:
def create_research_workflow(model_name: str, temperature: float, api_key: str, provider_config: dict = None) -> StateGraph:
    """Create LangGraph workflow for research agent"""
    
    # Create graph
    workflow = StateGraph(ResearchState)
    
    # Add node with all required parameters
    node = ResearchAgentNode(
        model_name=model_name,
        temperature=temperature,
        api_key=api_key,
        provider_config=provider_config
    )
    workflow.add_node("research_agent", node)
    
    # Set entry point
    workflow.set_entry_point("research_agent")
    
    # Compile
    graph = workflow.compile()
    
    return graph


print("✓ create_research_workflow() function defined")
print("  ⚠ Workflow will be created after loading M2-config.yaml")

✓ create_research_workflow() function defined
  ⚠ Workflow will be created after loading M2-config.yaml


## 8. Testing

### Test Case: Актуальная информация о технологии

In [13]:
# Test case: Educational research for LangGraph topic
test_topic = """Topic: Schema-guided reasoning (structured output with constrained decoding) in LLM applications

Context: Generating educational material for intermediate Python developers learning agent architectures.

Research objectives:
- Identify current best practices and real-world applications
- Locate practical examples suitable for teaching
- Find sources that explain "why" and "how", not just "what"

Depth needed: Comprehensive (material should support thorough understanding)
Target audience: Developers who need both theoretical understanding and practical skills"""

print(f"Test topic:\n{test_topic}")

Test topic:
Topic: Schema-guided reasoning (structured output with constrained decoding) in LLM applications

Context: Generating educational material for intermediate Python developers learning agent architectures.

Research objectives:
- Identify current best practices and real-world applications
- Locate practical examples suitable for teaching
- Find sources that explain "why" and "how", not just "what"

Depth needed: Comprehensive (material should support thorough understanding)
Target audience: Developers who need both theoretical understanding and practical skills


In [14]:
# ============================================================================
# LOAD CONFIGURATION
# ============================================================================
import yaml
import logging
from pathlib import Path
from langchain_openai import ChatOpenAI

# Configure logging for OpenAI client
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logging.getLogger("openai").setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.DEBUG)

print("✓ Debug logging enabled for OpenAI and HTTPX")

# Load M2 config
config_path = Path("M2-config.yaml")
if config_path.exists():
    with open(config_path) as f:
        m2_config = yaml.safe_load(f)
    print(f"✓ Loaded M2 configuration from {config_path}")
else:
    raise FileNotFoundError(f"M2-config.yaml not found at {config_path}")

# Resolve API keys
provider_config = m2_config.get("provider", {})
tavily_config = m2_config.get("tavily", {})

openai_api_key = resolve_api_key(provider_config.get("api_key", "$OPENAI_API_KEY"))
tavily_api_key = resolve_api_key(tavily_config.get("api_key", "$TAVILY_API_KEY"))

# Set environment variables for compatibility
os.environ["OPENAI_API_KEY"] = openai_api_key
os.environ["TAVILY_API_KEY"] = tavily_api_key

print(f"✓ Provider: {provider_config.get('name', 'openai')}")
print(f"✓ Base URL: {provider_config.get('base_url', 'default')}")
print(f"✓ OpenAI API key loaded: {openai_api_key[:8]}...")
print(f"✓ Tavily API key loaded: {tavily_api_key[:8]}...")

# Initialize Tavily client NOW (after keys are loaded)
tavily_client = TavilyClient(api_key=tavily_api_key)
print(f"✓ Tavily client initialized")

# Load model settings from config (NO DEFAULTS)
MODEL_NAME = m2_config["model"]
TEMPERATURE = m2_config["temperature"]

# Override test topic from config if exists
test_topic = m2_config.get("topic", "Schema-guided reasoning в LLM applications")

# Research settings
research_config = m2_config.get("research", {})
MAX_ITERATIONS = research_config.get("max_iterations", MAX_ITERATIONS)
MAX_SEARCHES = research_config.get("max_searches", MAX_SEARCHES)
MAX_EXTRACTIONS = research_config.get("max_extractions", MAX_EXTRACTIONS)

print(f"\n  Topic: {test_topic[:60]}...")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Max iterations: {MAX_ITERATIONS}")

# Create research workflow with ALL parameters from config
print(f"\n{'='*60}")
print("CREATING RESEARCH WORKFLOW")
print(f"{'='*60}")
research_graph = create_research_workflow(
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    api_key=openai_api_key,
    provider_config=provider_config
)
print("✓ Research workflow created and compiled")

✓ Debug logging enabled for OpenAI and HTTPX
✓ Loaded M2 configuration from M2-config.yaml
✓ Provider: openai
✓ Base URL: None
✓ OpenAI API key loaded: sk-proj-...
✓ Tavily API key loaded: tvly-dev...
✓ Tavily client initialized

  Topic: Schema-guided reasoning для разработчиков LLM-агентов...
  Model: gpt-4.1
  Temperature: 0.3
  Max iterations: 8

CREATING RESEARCH WORKFLOW
  ✓ LLM initialized: gpt-4.1 (temp=0.3)
✓ Research workflow created and compiled


/tmp/ipykernel_1205956/716066826.py:14: UserWarning: The 'config' parameter should be typed as 'RunnableConfig' or 'RunnableConfig | None', not '<class 'dict'>'. 
  workflow.add_node("research_agent", node)


In [15]:
# ============================================================================
# INTERCEPT RAW RESPONSE
# ============================================================================
import json

# Intercept at httpx level to see raw JSON response
import httpx

original_post = httpx.AsyncClient.post

async def debug_post(self, *args, **kwargs):
    url = args[0] if args else kwargs.get('url', '')
    
    # Only intercept OpenAI API calls
    if 'chat/completions' in str(url):
        print(f"\n{'='*80}")
        print("🔍 API REQUEST")
        print(f"{'='*80}")
        print(f"URL: {url}")
        
        # Print request body
        if 'json' in kwargs:
            request_body = kwargs['json']
            print(f"\nRequest body:")
            print(f"  Model: {request_body.get('model')}")
            print(f"  Temperature: {request_body.get('temperature')}")
            
            # Print response_format if present
            if 'response_format' in request_body:
                print(f"\n  Response Format:")
                print(json.dumps(request_body['response_format'], indent=2)[:2000])
    
    # Call original
    response = await original_post(self, *args, **kwargs)
    
    # Intercept response
    if 'chat/completions' in str(url):
        print(f"\n{'='*80}")
        print("🔍 API RESPONSE")
        print(f"{'='*80}")
        print(f"Status: {response.status_code}")
        
        # Try to decode response JSON
        try:
            response_json = response.json()
            print(f"\nResponse JSON:")
            print(json.dumps(response_json, indent=2, ensure_ascii=False)[:3000])
        except Exception as e:
            print(f"Failed to decode response: {e}")
    
    return response

httpx.AsyncClient.post = debug_post

print("✓ HTTP interception enabled")

# ============================================================================
# RUN RESEARCH AGENT
# ============================================================================

# Create initial state
initial_state = ResearchState(
    input_content=test_topic
)

# Run workflow
print("\n" + "="*70)
print("STARTING RESEARCH AGENT")
print("="*70)

final_state = await research_graph.ainvoke(initial_state)

print("\n" + "="*70)
print("RESEARCH COMPLETED")
print("="*70)

2025-11-25 21:23:59,518 - openai._base_client - DEBUG - Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Helper-Method': 'chat.completions.parse'}, 'files': None, 'idempotency_key': 'stainless-python-retry-8da09513-a57e-4267-ade3-54d9c448f21f', 'post_parser': <function AsyncCompletions.parse.<locals>.parser at 0x7f359ab965c0>, 'json_data': {'messages': [{'content': '<role>\nYou are a research assistant preparing source material for educational content generation.\nYour research will be used by a downstream system to create comprehensive learning materials.\n</role>\n\n<context>\nYour output feeds into an educational content generator that creates:\n- Structured learning materials (theory + practice)\n- Examples and exercises\n- Explanations suitable for students\n\nThe generator needs rich, authoritative information to work with.\n</context>\n\n<guidelines>\n- Prioritize EDUCATIONAL value: explanations, examples, practical applications\n- Seek au

2025-11-25 21:23:59,640 - httpcore.connection - DEBUG - connect_tcp.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x7f359aa04950>
2025-11-25 21:23:59,641 - httpcore.connection - DEBUG - start_tls.started ssl_context=<ssl.SSLContext object at 0x7f359ad3e720> server_hostname='api.openai.com' timeout=None


✓ HTTP interception enabled

STARTING RESEARCH AGENT

🤔 Selecting action (iteration 0)...


2025-11-25 21:23:59,704 - httpcore.connection - DEBUG - start_tls.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x7f359a9d80d0>
2025-11-25 21:23:59,706 - httpcore.http11 - DEBUG - send_request_headers.started request=<Request [b'POST']>
2025-11-25 21:23:59,708 - httpcore.http11 - DEBUG - send_request_headers.complete
2025-11-25 21:23:59,709 - httpcore.http11 - DEBUG - send_request_body.started request=<Request [b'POST']>
2025-11-25 21:23:59,711 - httpcore.http11 - DEBUG - send_request_body.complete
2025-11-25 21:23:59,712 - httpcore.http11 - DEBUG - receive_response_headers.started request=<Request [b'POST']>
2025-11-25 21:24:08,385 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Tue, 25 Nov 2025 18:24:09 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'

✓ Selected: D_GeneratePlanTool
✓ Research goal set: Gather authoritative, educationally valuable material on schema-guided reasoning for developers building LLM agents, covering theory, practical implementation, examples, and pitfalls.

🤔 Selecting action (iteration 1)...


...

  ✓ Extracted 20148 chars (stored: 5000)

🤔 Selecting action (iteration 5)...


2025-11-25 21:24:24,478 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Tue, 25 Nov 2025 18:24:25 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'user-88xyx2ymydeenhtkw13e8c33'), (b'openai-processing-ms', b'2965'), (b'openai-project', b'proj_l9lXGWTNjaFwPmwCYXCvIaao'), (b'openai-version', b'2020-10-01'), (b'x-envoy-upstream-service-time', b'2979'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'800000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'799465'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-tokens', b'40ms'), (b'x-request-id', b'req_79de8165a85842678fe9b796d57ffa06'), (b'x-openai-proxy-wasm', b'v0.1'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=31536000

✓ Selected: D_ReasoningTool
💭 Reasoning:
  - We have extracted a comprehensive definition and overview of schema-guided reasoning in LLM agents from source [1].
  - We have also extracted practical implementation examples from source [4], likely covering code, workflows, and real-world applications.
  - Together, these sources provide both theoretical background and practical guidance, which are essential for educational content.
  Enough data: True
  Task completed: False

🤔 Selecting action (iteration 6)...


2025-11-25 21:24:41,989 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Tue, 25 Nov 2025 18:24:42 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'user-88xyx2ymydeenhtkw13e8c33'), (b'openai-processing-ms', b'16684'), (b'openai-project', b'proj_l9lXGWTNjaFwPmwCYXCvIaao'), (b'openai-version', b'2020-10-01'), (b'x-envoy-upstream-service-time', b'16838'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'800000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'799357'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-tokens', b'48ms'), (b'x-request-id', b'req_abaa6c841ef746c8a89cd1ba736fcf20'), (b'x-openai-proxy-wasm', b'v0.1'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=315360

✓ Selected: D_CreateReportTool
✓ Report created: 3247 chars

RESEARCH COMPLETED


### Results Analysis

In [16]:
# Summary
print("\n📊 EXECUTION SUMMARY")
print("="*60)
print(f"Total iterations: {final_state['iteration']}")
print(f"Searches performed: {final_state['searches_used']}/{MAX_SEARCHES}")
print(f"Extractions performed: {final_state['extractions_used']}/{MAX_EXTRACTIONS}")
print(f"Sources collected: {len(final_state['sources'])}")
print(f"Final state: {final_state['agent_state']}")
print(f"Report length: {len(final_state['final_report'] or '')} chars")
print("="*60)


📊 EXECUTION SUMMARY
Total iterations: 7
Searches performed: 2/6
Extractions performed: 2/5
Sources collected: 5
Final state: completed
Report length: 3247 chars


In [17]:
# Sources list
print("\n📚 COLLECTED SOURCES")
print("="*60)

for url, source in final_state["sources"].items():
    print(f"\n[{source.number}] {source.title}")
    print(f"    URL: {source.url}")
    print(f"    Snippet: {source.snippet[:100]}...")
    if source.full_content:
        print(f"    Full content: {source.char_count} chars extracted")

print("\n" + "="*60)


📚 COLLECTED SOURCES

[1] [2502.03450] A Schema-Guided Reason-while-Retrieve framework ...
    URL: https://ar5iv.labs.arxiv.org/html/2502.03450
    Snippet: Two agents collaborate iteratively, enabling sequential reasoning and adaptive attention to graph in...
    Full content: 14429 chars extracted

[2] Schema-Guided Reasoning (SGR): Fixing Broken LLM Pipelines for ...
    URL: https://medium.com/@info_37025/schema-guided-reasoning-sgr-fixing-broken-llm-pipelines-for-measurable-results-b5b181bf32e6
    Snippet: # Schema-Guided Reasoning (SGR): Fixing Broken LLM Pipelines for Measurable Results Many enterprises...

[3] Schema-Guided Reasoning: What Changed in One Year
    URL: https://abdullin.substack.com/p/catching-up-a-years-worth-of-ai-developments
    Snippet: I also heard a report of a project that uses SGR DeepResearch core to drive internal knowledge minin...

[4] SGR Demo
    URL: https://abdullin.com/schema-guided-reasoning/demo
    Snippet: # Tool: Retrieves customer data s

In [18]:
# Final report
from IPython.display import Markdown, display

print("\n📝 FINAL REPORT")
print("="*60 + "\n")

if final_state["final_report"]:
    display(Markdown(final_state["final_report"]))
else:
    print("⚠ No report generated")


📝 FINAL REPORT



Schema-guided reasoning (SGR) is an advanced approach in the design and deployment of large language model (LLM) agents, where the agent's reasoning process is structured and constrained by explicit schemas—predefined templates or frameworks that guide how information is processed, interpreted, and acted upon [1]. This method contrasts with purely free-form or end-to-end reasoning, offering greater reliability, interpretability, and alignment with developer intent.

**Key Concepts**
- **Schema:** In SGR, a schema is a formal structure (often a JSON or similar data template) that defines the expected inputs, intermediate steps, and outputs for a reasoning task. Schemas can encode workflows, tool invocation patterns, or decision trees [1].
- **Guided Reasoning:** The LLM is prompted or constrained to reason within the bounds of the schema, ensuring that its outputs are well-structured, predictable, and easier to validate or post-process [1].
- **Applications:** SGR is especially valuable in tool-augmented agents, API orchestration, complex multi-step tasks, and scenarios demanding high reliability or compliance [1][4].

**Practical Example**
Suppose an LLM agent is designed to assist with travel booking. A schema might define the following reasoning steps:
1. **Extract user intent** (destination, dates, preferences)
2. **Search for flights** (using an external API)
3. **Present options** (structured as a list)
4. **Book selected flight** (with confirmation)

The schema would specify the expected data types and structure at each step. The LLM is prompted to fill in each part of the schema, ensuring no steps are skipped and all required information is collected [4].

**Implementation Patterns**
- **Prompt Engineering:** Prompts are crafted to instruct the LLM to reason according to the schema, often using explicit instructions or few-shot examples [1][4].
- **Schema Validation:** Outputs are validated against the schema, catching errors or omissions early [4].
- **Tool Integration:** Schemas can define when and how external tools or APIs are invoked, making agent behavior modular and auditable [4].

**Common Pitfalls and Best Practices**
- **Pitfall:** Overly rigid schemas can limit the flexibility and creativity of the LLM, while under-specified schemas may not provide enough guidance [1].
- **Best Practice:** Iteratively refine schemas based on observed agent behavior and user feedback [4].
- **Pitfall:** Schema drift—when the LLM's outputs diverge from the schema over time—can occur, especially with complex or ambiguous tasks [1].
- **Best Practice:** Use automated schema validation and fallback strategies to handle deviations [4].

**Educational Takeaways**
- SGR bridges the gap between free-form LLM reasoning and robust, production-ready agent workflows.
- Developers should balance schema specificity and flexibility, leveraging validation and modular design.
- Practical implementation involves prompt design, schema definition, and integration with external tools and validation systems.

**References**
[1] Authoritative overview and definition of schema-guided reasoning in LLM agents.
[4] Practical implementation examples, code snippets, and workflow patterns for SGR in agent development.

## 9. Validation

### Quality Checklist

**Agent Behavior:**
- [ ] Generated initial plan (iteration 0)
- [ ] Performed web searches with meaningful queries
- [ ] Collected sources with proper numbering
- [ ] Used reasoning steps for decision-making
- [ ] Created final report with inline citations

**Source Management:**
- [ ] Sources have sequential numbering [1], [2], [3]
- [ ] No duplicate sources (URL-based deduplication)
- [ ] Citations in report match source numbers
- [ ] Sources contain meaningful information

**Report Quality:**
- [ ] Comprehensive coverage of the topic
- [ ] Proper use of inline citations [1], [2]
- [ ] Logical structure and flow
- [ ] Accurate information from sources
- [ ] Self-sufficient content

**Resource Management:**
- [ ] Respected search limits (MAX_SEARCHES)
- [ ] Respected iteration limits (MAX_ITERATIONS)
- [ ] Made efficient use of resources
- [ ] Completed within reasonable time

## 10. Observations & Next Steps

### Key Observations

_(Fill after testing)_

**What works well:**
- 

**What needs improvement:**
- 

**Agent behavior patterns:**
- 

### Next Steps

**If MVP successful:**
1. [ ] Create `ResearchAgentNode` in `learnflow/nodes/research.py`
2. [ ] Add tool schemas to `learnflow/models/`
3. [ ] Integrate into main workflow before `planning_structure`
4. [ ] Add configuration to `configs/graph.yaml`
5. [ ] Test end-to-end через Telegram bot

**If improvements needed:**
1. [ ] Iterate on prompts in `_build_messages()`
2. [ ] Test with different topics (technical, current events, mathematical)
3. [ ] Tune resource limits (MAX_SEARCHES, MAX_ITERATIONS)
4. [ ] Improve reasoning quality
5. [ ] Better citation formatting

**Future extensibility:**
1. [ ] Add RAG search tool (Telegram channels)
2. [ ] Add document search tool (PDF, методички)
3. [ ] Implement BaseSourceTool abstraction
4. [ ] Add trigger logic (когда нужен research, когда нет)
5. [ ] HITL through interrupt() for clarifications

In [19]:
# ============================================================================
# SAVE OUTPUT FOR NEXT NOTEBOOK
# ============================================================================
import json
from pathlib import Path

# Create outputs directory
outputs_dir = Path(m2_config["outputs_dir"])
outputs_dir.mkdir(exist_ok=True)

# Prepare output data
output_data = {
    "topic": test_topic,
    "final_report": final_state["final_report"],
    "sources": {
        url: {
            "number": source.number,
            "title": source.title,
            "url": source.url,
            "snippet": source.snippet,
            "full_content": source.full_content,
            "char_count": source.char_count
        }
        for url, source in final_state["sources"].items()
    },
    "metadata": {
        "iterations": final_state["iteration"],
        "searches_used": final_state["searches_used"],
        "extractions_used": final_state["extractions_used"],
        "model": MODEL_NAME,
        "temperature": TEMPERATURE
    }
}

# Save to JSON
output_path = outputs_dir / m2_config["research_output"]
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"\n{'='*60}")
print(f"✓ Research output saved to: {output_path}")
print(f"  Topic: {output_data['topic'][:60]}...")
print(f"  Sources: {len(output_data['sources'])}")
print(f"  Report length: {len(output_data['final_report'])} chars")
print(f"{'='*60}")
print(f"\n▶ Next: Run M2-01-section-generation.ipynb")


✓ Research output saved to: outputs/research_output.json
  Topic: Schema-guided reasoning для разработчиков LLM-агентов...
  Sources: 5
  Report length: 3247 chars

▶ Next: Run M2-01-section-generation.ipynb
